# A3 - Problem 5.43 - Truck's Gas Consumption

This is a shortest path problem! The decision variables $x_{ij}$ take the value of 1 if the corresponding arc is selected (note that we don't need to define these as binary as we have a pure network!), and 0 otherwise. $L$ represents the set of cities and $A$ the set of arcs. In this particular case, as we want to find the shortest path between New York and Los Angeles, New York has a net demand of -1 (demand = 0, supply = 1), while Los Angeles has a net demand of 1 (demand of 1 and supply of 0). All the other cities in the network have both supplies and demands equal to 0.

\begin{equation*}
\begin{aligned}
& {\text{minimize}}
& & \sum_{(i,j) \in N}x_{ij}C_{ij} & \color{blue}{\longrightarrow \textbf{Fuel consumption minimization}}\\[2mm]
& \text{subject to}&&\\[2mm]
& & & \sum_{j \in L | (j,i) \in A}x_{ji}-\sum_{j \in L | (i,j) \in A}x_{ij} = (D_{i}-S_{i}), \text{ for } i \in L & \color{blue}{\longrightarrow \textbf{Flow balancing}}\\[2mm]
& & & x_{ij} \geq 0, \text{ for } (i,j) \in A & \color{blue}{\longrightarrow \textbf{Nonegative variables}}\\[2mm]
\end{aligned}
\end{equation*}

In [1]:
from gurobipy import *

In [2]:
location, supply, demand = multidict({
    'New_York':[1,0], 'Cleveland':[0,0], 'St_Louis':[0,0],
    'Nashville':[0,0], 'Phoenix':[0,0], 'Dallas':[0,0],
    'Salt_Lake_City':[0,0], 'Los_Angeles':[0,1]
})

arcs, cost = multidict({('New_York','Cleveland'): 400,  ('New_York','St_Louis'): 950, ('New_York','Nashvile'): 800,
     ('Cleveland','Phoenix'): 1800,  ('Cleveland','Dallas'): 900,  ('St_Louis','Phoenix'): 1100,
     ('St_Louis','Dallas'): 900,     ('Nashville','Dallas'): 600,  ('Nashville','Salt_Lake_City'): 1200,
     ('Phoenix','Los_Angeles'): 400, ('Dallas','Phoenix'): 900,   ('Dallas','Salt_Lake_City'): 1000,
     ('Dallas','Los_Angeles'): 1300, ('Salt_Lake_City','Los_Angeles'): 600})

In [3]:
shortestpath = Model("shortestpath")

Academic license - for non-commercial use only - expires 2022-08-04
Using license file C:\Users\novoadlj\gurobi.lic


In [4]:
flow = shortestpath.addVars(arcs, name="flow")
totalcost = shortestpath.setObjective(quicksum(flow[i,j]*cost[i,j] for (i,j) in arcs))

In [5]:
balance = shortestpath.addConstrs(quicksum(flow[j,i] for j in location if (j, i) in arcs)
                                 - quicksum(flow[i,j] for j in location if (i, j) in arcs) == (demand[i] - supply[i]) for i in location)

In [6]:
shortestpath.optimize()

Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 8 rows, 14 columns and 26 nonzeros
Model fingerprint: 0x225fe5b6
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 5 rows and 7 columns
Presolve time: 0.01s
Presolved: 3 rows, 7 columns, 14 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2500000e+03   2.000000e+00   0.000000e+00      0s
       1    2.4500000e+03   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds
Optimal objective  2.450000000e+03


In [7]:
# Display optimal production plan
for v in shortestpath.getVars():
    if v.x>0:
        print(v.varName, v.x)
    
print("Optimal total gas consumption:", "$"+str(shortestpath.objVal))

flow[New_York,St_Louis] 1.0
flow[St_Louis,Phoenix] 1.0
flow[Phoenix,Los_Angeles] 1.0
Optimal total gas consumption: $2450.0
